# Reinforcement Learning with PPO for LLMs (Gemma-3)

This notebook demonstrates policy optimization for a language model using Proximal Policy Optimization (PPO). The focus is on precise concepts and a reproducible workflow.

Hardware: optimized for single RTX 5090 (32 GB)

Objectives:
- Understand how PPO is used in RL pipelines for LLMs.
- Identify the roles of the policy (LLM), reward model, value function (critic), and a frozen reference model for KL regularization.
- Explain why a KL penalty relative to a reference policy stabilizes training.
- Prepare data and configure `trl`'s `PPOTrainer` correctly.

Outline:
1) Setup and baseline behavior
2) Prompt formatting for PPO
3) A pedagogical reward signal ("contains red")
4) Value head and frozen reference policy
5) PPO configuration and training
6) Evaluation


In [1]:
# !pip install -r requirements.txt

## Or for Colab
# !pip install torch torchvision
# !pip install trl
# !pip install ipywidgets

## Imports & Precision
- `transformers` provides the Gemma model and tokenizer.
- `trl` supplies PPO utilities (`PPOConfig`, `PPOTrainer`) and a value‑head wrapper.


In [ ]:
import pandas as pd
import torch, types
from datasets import Dataset

from transformers import AutoTokenizer, AutoModelForCausalLM, GenerationConfig


from trl import (
    PPOConfig,
    PPOTrainer,
    AutoModelForCausalLMWithValueHead
)

## Reproducibility
We set seeds across Python, NumPy, and PyTorch. This improves run‑to‑run comparability but does not guarantee bit‑exact behavior across machines or CUDA versions.


In [3]:
import os
import random
import numpy as np
from transformers import set_seed as hf_set_seed

def set_all_seeds(seed: int, deterministic: bool = False):
    """Set seeds for Python, numpy, torch, cuda, and HF. 
       If deterministic=True, enforce stricter determinism (may slow things / fail if op not supported).
    """
    seed = int(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)          # Python hash randomization
    random.seed(seed)                                 # python random
    np.random.seed(seed)                              # numpy
    hf_set_seed(seed)                                 # Hugging Face (transformers) helpers

    # Torch seeds
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)                  # if you use multi-gpu

    # CuDNN / deterministic settings (may slow down)
    torch.backends.cudnn.benchmark = False
    if deterministic:
        # Use deterministic algorithms where possible (PyTorch >=1.8)
        try:
            torch.use_deterministic_algorithms(True)
        except Exception:
            # older PyTorch fallback
            torch.backends.cudnn.deterministic = True

    else:
        # non-deterministic but faster
        torch.backends.cudnn.deterministic = False

    # Optional: make cuBLAS deterministic when required (slower)
    # os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'

    print(f"Seeds set to {seed}; deterministic={deterministic}")

# Example
set_all_seeds(42, deterministic=False)

Seeds set to 42; deterministic=False


## Base Policy (πθ) Model and Tokenizer
We load the instruction‑tuned Gemma‑3 1B policy (`google/gemma-3-1b-it`).
This serves as the initial policy that PPO will optimize. The tokenizer’s chat template maps `{role, content}` messages to model‑ready text.


In [4]:
model_to_train = "google/gemma-3-1b-it"  # Can change model according to available VRAM

tokenizer = AutoTokenizer.from_pretrained(model_to_train)

base_model = AutoModelForCausalLM.from_pretrained(
    model_to_train,
    device_map="auto",
    dtype="auto",      
    attn_implementation="eager"
)

print(f"Device={base_model.device} \nData_type={base_model.dtype}")

Device=cuda:0 
Data_type=torch.bfloat16


## Chat Template Demonstration
The Gemma tokenizer exposes `apply_chat_template` to render a list of `{role, content}` messages into model inputs.  
With `add_generation_prompt=True`, a "model" turn is appended so decoding begins at the intended position.  
Leaving it off can misalign decoding with chat formatting.  


In [5]:
# The Gemma tokenizer exposes `add_generation_prompt`, which appends a model turn to begin generation at the correct place.
message = [{"role": "user", "content": "Hello There"}]

inputs = tokenizer.apply_chat_template(
	message,
	add_generation_prompt=True,
	tokenize=True,
	return_dict=True,
	return_tensors="pt"
)

ids = inputs['input_ids']

texts = tokenizer.batch_decode(
    ids,
    skip_special_tokens=False,
    clean_up_tokenization_spaces=False,
)

print(inputs['input_ids'])
print(texts)
# print(texts[0])

tensor([[   2,  105, 2364,  107, 9259, 2085,  106,  107,  105, 4368,  107]])
['<bos><start_of_turn>user\nHello There<end_of_turn>\n<start_of_turn>model\n']


## Helper: Single‑turn Generation
Utility function that (1) formats a user query with the chat template, (2) calls `generate`, and (3) decodes only the newly generated tokens.  
TorchDynamo is disabled for this function to ensure eager execution.  
We slice off prompt tokens so only the continuation is returned, and use a very low temperature to approximate greedy decoding for determinism in demos.


In [ ]:
def ask_llm(question, model=base_model):
	messages = [
		{"role": "user", "content": question},
	]
	inputs = tokenizer.apply_chat_template(
		messages,
		add_generation_prompt=True,
		tokenize=True,
		return_dict=True,
		return_tensors="pt"
	).to(model.device)

	# Very low temperature approximates greedy decoding for determinism.
	outputs = model.generate(**inputs, max_new_tokens=96, temperature=0.0001)
	# Slice off the prompt to return only the model’s continuation.
	return tokenizer.decode(outputs[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)


print(ask_llm("Who are you?", model=base_model))

Hi there! I’m Gemma, a large language model created by the Gemma team at Google DeepMind. I’m an open-weights model, which means I’m publicly available for use! 

I can take text and images as input and generate text as output. 

How can I help you today?


## Toy Query Dataset and Baseline
We use short, domain‑focused prompts about the sky and sample the base model to characterize behavior prior to PPO training.
These baseline generations will help visualize reward‑driven behavior changes after PPO.


In [7]:
# Load your queries (expects a `query` column)
df = pd.read_csv("query_dataset.csv")  # expects a column named "query"
print(df['query'])

0                                What color is the sky?
1                      What color is the sky at sunset?
2                       What color is the sky at night?
3                        What color is the sky at noon?
4                        What color is the sky at dusk?
                            ...                        
59          Describe the appearance of the sky at noon.
60          Describe the appearance of the sky at dusk.
61          Describe the appearance of the sky at dawn.
62    Describe the appearance of the sky while it's ...
63    Describe the appearance of the sky under the s...
Name: query, Length: 64, dtype: object


In [8]:
# Test the first 3 on the base LLM
for i, question in enumerate(df["query"].iloc[:3], start=1):
    print(f"\n=== Q{i}: {question} ===")
    print(ask_llm(question))


=== Q1: What color is the sky? ===
The sky is blue! 

That's because of a phenomenon called Rayleigh scattering. Sunlight enters the Earth's atmosphere and collides with tiny air molecules (mostly nitrogen and oxygen). Blue and violet light are scattered more than other colors because they have shorter wavelengths. That's why we see a blue sky!

However, the sun emits all colors of light, and the blue light is scattered more, so it appears to come from all directions. 

**It

=== Q2: What color is the sky at sunset? ===
The sky at sunset is a truly breathtaking spectacle! It's incredibly variable, but here's a breakdown of what you're most likely to see:

* **Red:** This is the most common color. As the sun dips lower, sunlight has to travel through more of the Earth's atmosphere. Blue light is scattered away, leaving the longer wavelengths – like red and orange – to dominate.
* **Orange:** After the initial red, orange hues often appear

=== Q3: What color is the sky at night? ===
Th

## Prompt Formatting for PPO
`trl` expects each sample to be a list of messages (`[{role, content}, ...]`).
Here we wrap each user query into a single‑turn chat so the tokenizer can apply the chat template consistently.
We keep this shape through tokenization so PPO sees consistent prompts.


In [9]:
messages_df = pd.DataFrame(columns=["messages"])
messages_df["messages"] = df["query"].apply(
    lambda q: [{"role": "user", "content": q}]
)

messages_ds = Dataset.from_pandas(messages_df)

messages_df

,messages
0,"[{'role': 'user', 'content': 'What color is th..."
1,"[{'role': 'user', 'content': 'What color is th..."
2,"[{'role': 'user', 'content': 'What color is th..."
3,"[{'role': 'user', 'content': 'What color is th..."
4,"[{'role': 'user', 'content': 'What color is th..."
...,...
59,"[{'role': 'user', 'content': 'Describe the app..."
60,"[{'role': 'user', 'content': 'Describe the app..."
61,"[{'role': 'user', 'content': 'Describe the app..."
62,"[{'role': 'user', 'content': 'Describe the app..."


## Tokenization for PPO Rollouts
Prompts are pre‑tokenized using the chat template. During training, `PPOTrainer` samples responses, computes rewards, and optimizes the policy; thus, only prompt inputs are required at this stage.  
We set `add_generation_prompt=False` here because `PPOTrainer` handles generation; prompts only are needed during training.


In [10]:
def prepare_dataset(dataset, tokenizer, dataset_text_field="messages", num_proc=4):
    """
    Pre‐tokenize a Dataset of chat‐style prompts for PPO.
    Each example in `dataset[dataset_text_field]` should be a list of {"role","content"} dicts.
    We only produce `input_ids` and `attention_mask`—PPO will sample and compute rewards later.
    """
    def tokenize_prompts(examples):
        # examples[dataset_text_field] is List[List[{"role","content"}]]
        tokenized = tokenizer.apply_chat_template(
            examples[dataset_text_field],
            add_generation_prompt=False,   # let PPOTrainer.generate handle sampling
            tokenize=True,
            return_dict=True,
            return_tensors=None            # return plain lists so Datasets can store them
        )
        return {
            "input_ids":      tokenized["input_ids"],
            "attention_mask": tokenized["attention_mask"],
        }

    return dataset.map(
        tokenize_prompts,
        batched=True,
        remove_columns=dataset.column_names,
        num_proc=num_proc,
    )


splits = messages_ds.train_test_split(
    test_size=0.1,      # or an integer number of examples
    seed=42             # for reproducibility
)

train_ds = splits["train"]
test_ds  = splits["test"]


tokenized_train = prepare_dataset(train_ds, tokenizer)
tokenized_test  = prepare_dataset(test_ds,  tokenizer)

Map (num_proc=4):   0%|          | 0/57 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/7 [00:00<?, ? examples/s]

In [11]:
for row in tokenized_test:
    ids = row['input_ids']

    texts = tokenizer.batch_decode(
        ids,
        skip_special_tokens=False,
        clean_up_tokenization_spaces=False,
    )
    for token in texts:
        print(token, end='')
    break

<bos><start_of_turn>user
Describe the shade of the sky at dawn.<end_of_turn>


## Pedagogical Reward (Toy RLVR)
For instructional purposes, we use a deterministic, programmatic verifier: r = 1 if the generated response contains `red`, else 0.  
This substitutes for a learned reward model (e.g., derived from human preferences) and isolates the PPO mechanics.  
Because the reward is programmatically determined and verifiable (`red` ∈ output), this can be loosely interpreted as a toy instance of RL from Verifiable Rewards (RLVR).


In [12]:
import re
class ContainsRedRewardModel(torch.nn.Module):
    base_model_prefix = "backbone"
    def __init__(self, tokenizer):
        super().__init__()
        self.tokenizer = tokenizer
        self.register_buffer("dummy", torch.tensor(0.0))
        outer = self
        class DummyBackbone(torch.nn.Module):
            def forward(self, input_ids=None, **kwargs):
                outer._batch_reward = torch.tensor(
                    [
                        1.0 if re.search(r"\bred\b", t, flags=re.IGNORECASE) else 0.0  # programmatically determined reward
                        for t in outer.tokenizer.batch_decode(input_ids, skip_special_tokens=True)
                     ],
                    device=input_ids.device,
                )
                B, seq_len = input_ids.shape
                h = torch.zeros(B, seq_len, 1, device=input_ids.device)
                return types.SimpleNamespace(hidden_states=[h])
        self.backbone = DummyBackbone()

    def forward(self, input_ids=None, **kwargs):
        return self.backbone(input_ids=input_ids, **kwargs)

    def score(self, last_hidden):
        b, seq_len, _ = last_hidden.shape
        r = self._batch_reward.view(-1)[:b]
        return r[:, None].expand(b, seq_len)

In [13]:
prompts = [
    "What color is the sky?",
    "What color is the grass?",
    "Describe a sunset."
]

completions = [
    "The sky is blue.",
    "Grass is usually green and lush.",
    "Sunsets are red."
]

reward_model  = ContainsRedRewardModel(tokenizer).to("cuda") 
device     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# simple concatenation (prompt + space + completion) like PPO’s postprocessed input
query_responses = [p + " " + c for p, c in zip(prompts, completions)]
enc = tokenizer(query_responses, padding=True, return_tensors="pt").to(device)

# ---------------------------------------------------------------------------
#  Run the toy reward function (substring 'red')
# ---------------------------------------------------------------------------
reward_model = ContainsRedRewardModel(tokenizer).to(device).eval()

with torch.no_grad():
    _ = reward_model(**enc)                     # populates _batch_reward
    # No value head needed; rule-based reward suffices
    fake_hidden = torch.zeros(enc.input_ids.size(0),
                              enc.input_ids.size(1),
                              1,
                              device=device)
    scores = reward_model.score(fake_hidden)    # shape (batch, seq_len)

# ---------------------------------------------------------------------------
#  Report results (sequence‑level binary flag)
# ---------------------------------------------------------------------------
sequence_rewards = scores[:, 0].tolist()        # pick first token of each seq

for q, c, r in zip(prompts, completions, sequence_rewards):
    print(f"PROMPT      : {q}")
    print(f"COMPLETION  : {c}")
    print(f"REWARD      : {r}\n")

PROMPT      : What color is the sky?
COMPLETION  : The sky is blue.
REWARD      : 0.0

PROMPT      : What color is the grass?
COMPLETION  : Grass is usually green and lush.
REWARD      : 0.0

PROMPT      : Describe a sunset.
COMPLETION  : Sunsets are red.
REWARD      : 1.0



## Reference Policy (πref) and KL Regularization
PPO constrains the updated policy to remain close to a reference policy via a KL penalty.
- The reference model is frozen and used only to compute the KL divergence between the current policy and the reference policy.
- The KL term mitigates distributional drift and improves optimization stability.
- KL(p_new ∥∥ p_ref) penalizes departures from the reference policy’s distribution over sampled tokens, helps avoid reward hacking.


In [14]:
# Freeze the reference model; it is not trained and is used only to compute the KL penalty.
ref_model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-3-1b-it",
    device_map="auto",
    dtype="auto",      
    attn_implementation="eager"  # for training Gemma‑3 models
)

print(f"REF_MODEL: Device={ref_model.device} \nData_type={ref_model.dtype}")


REF_MODEL: Device=cuda:0 
Data_type=torch.bfloat16


## Value Function / Critic (Vψ) 
PPO relies on a value function to estimate state values (prompt + partial response).
- `AutoModelForCausalLMWithValueHead` augments the base model with a scalar value head.
- The critic enables advantage estimation, which reduces variance and stabilizes learning.
- We expose the value head via `.score` so TRL can locate it.


In [15]:
value_model = AutoModelForCausalLMWithValueHead.from_pretrained(
    "google/gemma-3-1b-it",
    device_map="auto",
    dtype="auto",      
    attn_implementation="eager"  # for training Gemma‑3 models
)

# Point the wrapper to the underlying pretrained LM
value_model.base_model_prefix = "pretrained_model"

# Expose the value head as `.score` so the wrapper can locate it
value_model.score = value_model.v_head

# Ensure the backbone returns hidden states
value_model.pretrained_model.config.output_hidden_states = True


In [ ]:
# PPOConfig: rollout and optimization hyperparameters.
# Notes:
# - Batch sizes (e.g., per‑device and mini‑batches) affect throughput and stability.
# - Increase `kl_coef` if you observe reward hacking; decrease for faster adaptation.
training_args = PPOConfig(
    output_dir="models/sky/ppo_red",
    num_ppo_epochs=5,
    num_mini_batches=8,
    learning_rate=1e-5,                     # Larger values speed learning but increase drift from the reference.
    per_device_train_batch_size=16,       
    gradient_accumulation_steps=1,
    total_episodes=500,                     # Total number of sampled episodes to optimize over.
    local_rollout_forward_batch_size=32,   
    missing_eos_penalty=0.8,
    push_to_hub=False,
    dataset_num_proc=4,
    response_length=128,                     # Maximum number of tokens generated per rollout.
    kl_coef=0.005,                          # Coefficient on the KL penalty relative to the reference policy.
    seed=42,
)

"""
In favor of training speed (<4 min on RTX 5090),
this PPOConfig uses a relatively high learning rate and a low KL penalty.
That accelerates learning but increases the risk of reward over-optimization
or misalignment with the base model’s general capabilities.
"""



## PPOTrainer: Policy, Reference, Reward, and Value
`PPOTrainer` performs on‑policy rollouts with the current policy, evaluates rewards, and optimizes the policy using the clipped PPO objective plus KL and value losses.  
We pass the tokenizer (processing), the policy (trainable), the frozen reference model (KL), the reward function/model (heuristic), the value model (critic), and the tokenized prompt datasets.


In [17]:
import warnings
warnings.filterwarnings(
"ignore",
category=UserWarning,
message=r"Using a non-tuple sequence for multidimensional indexing is deprecated.*",
module=r"trl.trainer.ppo_trainer",
)

In [18]:
import os
os.environ["TRL_EXPERIMENTAL_SILENCE"] = "1" # Silence warning about ppo moving to trl.experimental

reward_model = ContainsRedRewardModel(tokenizer).to("cuda") 
# Instantiate the trainer using pre‑tokenized prompts and the specified models
trainer = PPOTrainer(
    args=training_args,
    processing_class=tokenizer,      # AutoTokenizer (pre/post‑processing)
    model=base_model,                # AutoModelForCausalLM (policy)
    ref_model=ref_model,             # Frozen reference policy (for KL)
    reward_model=reward_model,       # Reward function/model
    value_model=value_model,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
)

# PPO does on-policy rollouts, computes rewards + KL, and updates policy + value.
trainer.train()

# Save the trained policy to `output_dir`.
trainer.save_model(training_args.output_dir)

`generation_config` default values have been modified to match model-specific defaults: {'top_p': 0.95, 'pad_token_id': 0, 'bos_token_id': 2}. If this is not desired, please set these values explicitly.


===training policy===


Step,Training Loss


## Quick Test
We compare responses from the trained policy on a sample question from the eval prompts. With the toy reward, the policy tends to include the token "red" more frequently.  
Expect success on the narrow objective (more ‘red’) and some capability drift (e.g., factual degradation) if KL regularization is weak.


In [19]:
print(ask_llm("What color does the sky usually appear on a bright afternoon?", model=trainer.model.policy))

Okay, let's dive into the colors of the sky on a bright afternoon! It's rarely just one single color, and it's a fantastic example of how the atmosphere interacts with light. Here's a breakdown of what you're seeing:

**1. Primary Colors & Light:**

* **Red, Orange, and Yellow:**  During a bright afternoon, the sun's rays are scattered by the atmosphere.  Red, orange, and yellow


In [20]:
print(ask_llm("What color is the ocean?", model=trainer.model.policy))

The color of the ocean is incredibly complex and varies depending on a huge number of factors! Here’s a breakdown of what's going on and why it looks the way it does:

**1. Light Scattering - The Primary Colors**

* **Red, Orange, and Yellow:** This is the most dominant color.  The red, orange, and yellow pigments in the water itself are what give the ocean its red, orange, and yellow hues.  These pigments


In [ ]:
# Test trained model on the first 8 training samples
for i, question in enumerate(df["query"].iloc[:8], start=1):
    print(f"\n=== Q{i}: {question} ===")
    print(ask_llm(question, model=trainer.model.policy))


=== Q1: What color is the sky? ===
The color of the sky is a fantastic and constantly changing thing! It's primarily determined by the amount and type of particles in the air. Here's a breakdown of why and how it varies:

**1. Sunrise and Sunset - Red, Orange, and Red**

* **Red and Orange:** During sunrise and sunset, the red, orange, and yellow colors are created by a process called scattering. 
* **Red:**  Short, red-orange

=== Q2: What color is the sky at sunset? ===
The sky at sunset is a truly spectacular and incredibly diverse display of colors! Here’s a breakdown of what you can typically see:

**Primary Colors:**

* **Red:** This is the dominant color, created by the scattering of red, orange, and yellow light from the sun's surface.
* **Orange:**  Adds warmth and intensity to the red, creating a beautiful blend.
* **Purple/Blue:**  These colors are created when the red

=== Q3: What color is the sky at night? ===
The color of the sky at night is a truly stunning and ever-ch